In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from keras.layers import InputLayer
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

SEED = 29

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
df = pd.read_parquet("train.parquet")

In [3]:
df.describe()

,seq_ix,step_in_seq,need_prediction,0,1,2,3,4,5,6,...,22,23,24,25,26,27,28,29,30,31
count,517000.00000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,...,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000
mean,258.00000,499.500000,0.899000,0.001247,0.022680,-0.006079,-0.013578,-0.001519,0.006435,-0.022507,...,0.021797,-0.015975,-0.010245,0.003196,-0.022505,-0.021408,0.021334,-0.007824,0.006495,-0.018510
std,149.24491,288.675269,0.301329,0.985162,0.994129,1.008178,1.010985,0.990943,0.989856,1.001293,...,0.989687,0.989816,0.991879,0.997269,0.997177,0.996491,0.995721,0.996676,1.002600,1.001944
min,0.00000,0.000000,0.000000,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,...,-5.199338,-3.400935,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338
25%,129.00000,249.750000,1.000000,-0.672111,-0.655956,-0.680784,-0.703100,-0.684677,-0.668105,-0.709381,...,-0.666198,-0.696765,-0.671657,-0.679927,-0.688384,-0.697000,-0.655391,-0.694185,-0.658455,-0.684046
50%,258.00000,499.500000,1.000000,-0.003134,0.013809,-0.006287,-0.020027,-0.003576,0.009730,-0.018166,...,0.030758,-0.010643,-0.006489,0.011550,-0.013160,-0.004827,0.017762,-0.012549,0.008081,-0.012764
75%,387.00000,749.250000,1.000000,0.655925,0.698124,0.663186,0.661700,0.670718,0.685277,0.669402,...,0.687280,0.663009,0.650527,0.684317,0.651541,0.652707,0.700777,0.669457,0.680425,0.643227
max,516.00000,999.000000,1.000000,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,...,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338


In [4]:
seqs = np.zeros((517,1000,33))
for i in range(516):
   seqs[i] = df[df["seq_ix"] == i].drop(columns= ["seq_ix","step_in_seq"]).values

# seqs[i][j][0] gives ith sequence jth row need_prediction


In [5]:
input_to_encoder_seqs = np.zeros((517,100,32))
for i in range(516):
   input_to_encoder_seqs[i] = df[(df["seq_ix"] == i) & (df["need_prediction"]==0)].drop(columns= ["need_prediction","seq_ix","step_in_seq"]).iloc[:-1].values

In [6]:
output_of_decoder_seqs = np.zeros((517,899,32))
for i in range(516):
   output_of_decoder_seqs[i] = df[(df["seq_ix"] == i) & (df["need_prediction"]==1)].drop(columns= ["need_prediction","seq_ix","step_in_seq"]).values

In [7]:
class Preprocessor:
    def __init__(self):
        self.scaler = MinMaxScaler()
    def fit(self,X_train):
        X = X_train.reshape(-1,X_train.shape[-1])
        #print(X.shape)
        X = self.scaler.fit_transform(X).reshape(X_train.shape)
        return X
    def transform(self,X_test):
        X = X_test.reshape(-1,X_test.shape[-1])
        return self.scaler.transform(X).reshape(X_test.shape)

In [8]:
preproc = Preprocessor()
X_train_temp ,X_test_temp, y_train , y_test = train_test_split(input_to_encoder_seqs,output_of_decoder_seqs,random_state=SEED,train_size=0.8)
X_train = preproc.fit(X_train_temp)
X_test = preproc.transform(X_test_temp)

In [13]:
class AutoEncoder(Model):
  def __init__(self,input_shape):
    self.num_neurons = [64,24]
    self.latent_dims = 25
    self.to_predict = 899
    super(AutoEncoder,self).__init__()
    self.encoder = tf.keras.Sequential(
        [
            InputLayer(input_shape),
            layers.LSTM(self.num_neurons[0],return_sequences=True),
            layers.LSTM(self.num_neurons[1],return_sequences=False,recurrent_dropout = 0.4),
            layers.Dropout(0.2),
            layers.Dense(self.latent_dims,activation = 'linear')
        ]
    )

    self.decoder = tf.keras.Sequential(
        [  
          InputLayer(shape = (self.latent_dims,)),
          layers.RepeatVector(self.to_predict),
          layers.LSTM(10,return_sequences=True,activation='tanh', recurrent_dropout= 0.7),
          layers.LSTM(input_shape[1],return_sequences=True,activation='tanh'),
          layers.TimeDistributed(layers.Dense(32,activation = "linear"))
        ]
        )

  def call(self,x):
    latent_vec = self.encoder(x)
    return self.decoder(latent_vec)

  def train(self,X_train,y_train):
    optim = tf.keras.optimizers.Adam(learning_rate=0.01)
    self.compile(optimizer = optim,loss= "mse")
    self.summary()
    self.fit(X_train,y_train,batch_size = 32, epochs = 15,verbose = True, validation_split = 0.2)
    


In [16]:
output_of_decoder_seqs.shape

(517, 899, 32)

In [14]:
autoenc = AutoEncoder([100,32])
autoenc.train(X_train = input_to_encoder_seqs,y_train=output_of_decoder_seqs)


Model: "auto_encoder_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_4 (Sequential)       │ (None, 25)             │        34,001 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_5 (Sequential)       │ (None, 899, 32)        │         8,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,001 (164.07 KB)

 Trainable params: 42,001 (164.07 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 59s 2s/step - loss: 0.9092 - val_loss: 0.8985
Epoch 2/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 646ms/step - loss: 0.8562 - val_loss: 0.8737
Epoch 3/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 606ms/step - loss: 0.8390 - val_loss: 0.8639
Epoch 4/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 581ms/step - loss: 0.8370 - val_loss: 0.8592
Epoch 5/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 572ms/step - loss: 0.8344 - val_loss: 0.8555
Epoch 6/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 579ms/step - loss: 0.8164 - val_loss: 0.8477
Epoch 7/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 583ms/step - loss: 0.8071 - val_loss: 0.8476
Epoch 8/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 577ms/step - loss: 0.8051 - val_loss: 0.8437
Epoch 9/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 586ms/step - loss: 0.7988 - val_loss: 0.8368
Epoch 10/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 578ms/step - loss: 0.7959 - val_loss: 0.8323
Epoch 11/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 571ms/step - loss: 0.7950 - val_loss: 0.8333
Epoch 12/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 592ms/step -